# INGEST

In [ ]:
# Import necessary modules
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df = pd.read_csv("Dataset-RR.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
# print out all null values
df.isnull().sum()

In [ ]:
df = df.dropna(subset=["Cuisines"])

In [ ]:
df.info()

In [ ]:
df.describe()

# EDA

In [ ]:
# Visualization of the dataset
cuisines = df["Cuisines"].value_counts().head(10)
cuisines.plot(kind="bar", color="skyblue")
plt.title("Top most Common Cuisines")
plt.xlabel("Cuisine")
plt.ylabel("Number of Restaurants")
plt.show()

In [ ]:
# Price Range Distribution
df["Price range"].value_counts().plot(
    kind="pie", autopct="%1.1f%%", colors=["gold", "lightblue", "lightgreen", "pink"]
)
plt.title("Price Range Distribution")
plt.ylabel("")
plt.show()

In [ ]:
# Rating distribution
df["Aggregate rating"].plot(kind="hist", bins=10, color="purple", edgecolor="black")
plt.title("Rating Distribution")
plt.xlabel("Aggregate rating")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Create Heatmap check for relationships betweeen Votes, Aggregate rating, Price-range
sns.heatmap(
    df[["Votes", "Aggregate rating", "Price range"]].corr(), annot=True, cmap="coolwarm"
)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# List of Useful features for the Restaurant Recommendation
useful_features = df[["City", "Cuisines", "Price range", "Aggregate rating", "Votes"]]
useful_features.head()

In [ ]:
# Identify Categorical and Numerical Features

categorical_features = useful_features.select_dtypes(include=["object"]).columns
numerical_features = useful_features.select_dtypes(exclude=["object"]).columns
numerical_weight =[2.0, 0.5, 1.0]
# Preprocessing for Categorical Features
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# Preprocessing for Numerical Features
numerical_transformer = StandardScaler()

# Combine both transformations using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [ ]:
# Apply the transformations
pipeline = Pipeline(steps=[("preprocessor", preprocessor)])

# Now we can fit and transform the data
X_transformed = pipeline.fit_transform(useful_features)

# MODEL

In [ ]:
# Compute the similarity matrix
similarity_matrix = cosine_similarity(X_transformed)

In [ ]:
# Adding the Restaurant ID and Restaurant Name to the useful_features
df = df.reset_index(drop=True)
useful_features = useful_features.reset_index(drop=True)

useful_features["Restaurant ID"] = df["Restaurant ID"]
useful_features["Restaurant Name"] = df["Restaurant Name"]

In [ ]:
def recommend_restaurants(
    restaurant_id, df, X_transformed, similarity_matrix, top_n=5
):
    """
    Recommend restaurants similar to a given restaurant.

    Parameters:
        restaurant_id (int): The ID of the restaurant for which recommendations are needed.
        df (pd.DataFrame): The original DataFrame containing restaurant details.
        X_transformed (np.array): The transformed data used for similarity calculations.
        similarity_matrix (np.array): Precomputed similarity matrix.
        top_n (int): Number of recommendations to return (default is 5).

    Returns:
        pd.DataFrame: DataFrame containing recommended restaurants with details.
    """
    # Get the index of the given restaurant
    try:
        index = df[df["Restaurant ID"] == restaurant_id].index[0]
    except IndexError:
        return f"Restaurant ID {restaurant_id} not found in the dataset."

    # Compute similarity scores for the restaurant
    similarity_scores = list(enumerate(similarity_matrix[index]))

    # Sort by similarity scores in descending order
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    # Get the indices of the top N most similar restaurants (exclude the input restaurant itself)
    recommend_indices = [i[0] for i in similarity_scores[1 : top_n + 1]]

    # Return the recommended restaurants with their details
    return df.iloc[recommend_indices][
        ["Restaurant Name", "Cuisines", "Aggregate rating"]
    ]

In [ ]:
# Example: Recommend restaurants for a specific Restaurant ID
sample_restaurant_id = 6318506  # Replace with an actual Restaurant ID from your dataset
recommendations = recommend_restaurants(
    sample_restaurant_id, useful_features, X_transformed, similarity_matrix, top_n=5
)

# Display the recommendations
print("Recommended Restaurants:")
print(recommendations)

# CONCLUSION

This project successfully developed a personalized restaurant recommendation system using content-based filtering. The model performed well in tailoring suggestions based on user preferences, providing significant value for users seeking personalized dining experiences. While the system performs effectively, future enhancements such as integrating collaborative filtering, handling the cold-start problem, and deploying the system for real-time recommendations can further improve its accuracy and scalability.